<a href="https://colab.research.google.com/github/AnaghaSreenath/gen-ai/blob/main/gai6(hallucination).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q pandas==2.2.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 83.0 MB/s eta 0:00:00


In [3]:
import pandas as pd
import torch
import faiss
import gradio as gr

print("Pandas version:", pd.__version__)
print("GPU available:", torch.cuda.is_available())
print("All imports successful!")

Pandas version: 2.2.3
GPU available: True
All imports successful!


In [4]:
from google.colab import files

uploaded = files.upload()

pdf_files = [
    name for name in uploaded.keys()
    if name.lower().endswith(".pdf")
]

if not pdf_files:
    raise ValueError("Please upload a PDF file.")

PDF_PATH = pdf_files[0]

print("Uploaded PDF:", PDF_PATH)

Saving sample_rag_document.pdf to sample_rag_document.pdf
Uploaded PDF: sample_rag_document.pdf


In [5]:
import fitz

document = fitz.open(PDF_PATH)
pages = []

for page_number, page in enumerate(document, start=1):
    text = page.get_text("text").strip()

    if text:
        pages.append({
            "page": page_number,
            "text": text
        })

document.close()

if not pages:
    raise ValueError(
        "No selectable text found in the PDF. It may need OCR."
    )

print("Pages containing text:", len(pages))
print("\nPDF preview:\n")
print(pages[0]["text"][:1000])

Pages containing text: 1

PDF preview:

Introduction to Artificial Intelligence
Artificial Intelligence (AI) is the field of computer science concerned with creating systems that can
perform tasks that normally require human intelligence.
Machine Learning
Machine Learning is a subset of Artificial Intelligence. It enables computers to learn patterns from
data and make predictions or decisions without being explicitly programmed for every task.
Deep Learning
Deep Learning is a branch of machine learning that uses neural networks with multiple layers. It is
widely used for image recognition, speech processing, natural language processing, and
recommendation systems.
Natural Language Processing
Natural Language Processing (NLP) allows computers to understand, process, and generate
human language. Applications include chatbots, translation systems, sentiment analysis, and
document question answering.
Retrieval-Augmented Generation
Retrieval-Augmented Generation (RAG) combines information r

In [6]:
CHUNK_SIZE = 180
CHUNK_OVERLAP = 40

chunks = []

for page in pages:
    words = page["text"].split()
    start = 0

    while start < len(words):
        end = min(start + CHUNK_SIZE, len(words))

        chunk_text = " ".join(words[start:end]).strip()

        if chunk_text:
            chunks.append({
                "page": page["page"],
                "text": chunk_text
            })

        if end == len(words):
            break

        start = end - CHUNK_OVERLAP

if not chunks:
    raise ValueError("No chunks were created.")

print("Total chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0]["text"])

Total chunks: 2

First chunk:

Introduction to Artificial Intelligence Artificial Intelligence (AI) is the field of computer science concerned with creating systems that can perform tasks that normally require human intelligence. Machine Learning Machine Learning is a subset of Artificial Intelligence. It enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task. Deep Learning Deep Learning is a branch of machine learning that uses neural networks with multiple layers. It is widely used for image recognition, speech processing, natural language processing, and recommendation systems. Natural Language Processing Natural Language Processing (NLP) allows computers to understand, process, and generate human language. Applications include chatbots, translation systems, sentiment analysis, and document question answering. Retrieval-Augmented Generation Retrieval-Augmented Generation (RAG) combines information retrieval 

In [7]:
CHUNK_SIZE = 180
CHUNK_OVERLAP = 40

chunks = []

for page in pages:
    words = page["text"].split()
    start = 0

    while start < len(words):
        end = min(start + CHUNK_SIZE, len(words))

        chunk_text = " ".join(words[start:end]).strip()

        if chunk_text:
            chunks.append({
                "page": page["page"],
                "text": chunk_text
            })

        if end == len(words):
            break

        start = end - CHUNK_OVERLAP

if not chunks:
    raise ValueError("No chunks were created.")

print("Total chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0]["text"])

Total chunks: 2

First chunk:

Introduction to Artificial Intelligence Artificial Intelligence (AI) is the field of computer science concerned with creating systems that can perform tasks that normally require human intelligence. Machine Learning Machine Learning is a subset of Artificial Intelligence. It enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task. Deep Learning Deep Learning is a branch of machine learning that uses neural networks with multiple layers. It is widely used for image recognition, speech processing, natural language processing, and recommendation systems. Natural Language Processing Natural Language Processing (NLP) allows computers to understand, process, and generate human language. Applications include chatbots, translation systems, sentiment analysis, and document question answering. Retrieval-Augmented Generation Retrieval-Augmented Generation (RAG) combines information retrieval 

In [8]:
import faiss
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS index created successfully!")
print("Indexed chunks:", index.ntotal)
print("Embedding dimension:", dimension)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index created successfully!
Indexed chunks: 2
Embedding dimension: 384


In [9]:
def retrieve_context(question, top_k=5):
    question = question.strip()

    if not question:
        return []

    top_k = max(1, min(int(top_k), index.ntotal))

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        question_embedding,
        top_k
    )

    retrieved = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        retrieved.append({
            "text": chunks[int(idx)]["text"],
            "page": chunks[int(idx)]["page"],
            "similarity": float(score)
        })

    return retrieved


test_results = retrieve_context(
    "What is the main topic of this document?",
    top_k=3
)

for i, item in enumerate(test_results, start=1):
    print(f"\nResult {i} | Page {item['page']}")
    print("Similarity:", round(item["similarity"], 3))
    print(item["text"][:400])


Result 1 | Page 1
Similarity: 0.32
provides that information to a language model to generate an answer. PDF-Based RAG A PDF-based RAG system extracts text from PDF documents, divides the text into smaller chunks, converts the chunks into numerical embeddings, and stores them in a vector database. When a user asks a question, the system retrieves the most relevant chunks and provides them to the language model as context. Advantages

Result 2 | Page 1
Similarity: 0.2
Introduction to Artificial Intelligence Artificial Intelligence (AI) is the field of computer science concerned with creating systems that can perform tasks that normally require human intelligence. Machine Learning Machine Learning is a subset of Artificial Intelligence. It enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for ever


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.eval()

print("Local model loaded successfully!")
print("GPU available:", torch.cuda.is_available())
print("Model device:", model.device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Local model loaded successfully!
GPU available: True
Model device: cuda:0


In [11]:
def local_generate(messages, max_new_tokens=250):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=6000
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output_ids[
        0, inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


# Test the model
test_messages = [
    {
        "role": "user",
        "content": "Explain RAG in one sentence."
    }
]

print(local_generate(test_messages, max_new_tokens=60))

RAG stands for "Reinventing Government Architecture," which is the process of redesigning and modernizing government systems to enhance efficiency, effectiveness, and user experience.


In [12]:
def generate_answer(question, retrieved):
    if not retrieved:
        return (
            "I could not find relevant information in the PDF.",
            ""
        )

    context = "\n\n".join(
        f"[Page {item['page']}]\n{item['text']}"
        for item in retrieved
    )

    messages = [
        {
            "role": "system",
            "content": (
                "Answer using only the supplied PDF context. "
                "Do not invent facts. If the answer is unavailable "
                "in the context, say so clearly."
            )
        },
        {
            "role": "user",
            "content": (
                f"PDF CONTEXT:\n{context}\n\n"
                f"QUESTION:\n{question}\n\n"
                "Answer clearly and concisely."
            )
        }
    ]

    answer = local_generate(
        messages,
        max_new_tokens=250
    )

    return answer, context


def rag_answer(question, top_k=5):
    retrieved = retrieve_context(question, top_k)

    answer, context = generate_answer(
        question,
        retrieved
    )

    return {
        "question": question,
        "answer": answer,
        "context": context,
        "retrieved": retrieved
    }


# Test the complete RAG pipeline
result = rag_answer(
    "What is a PDF-based RAG system?",
    top_k=5
)

print("QUESTION:", result["question"])
print("\nANSWER:\n", result["answer"])
print("\nRETRIEVED EVIDENCE:\n", result["context"][:1200])

QUESTION: What is a PDF-based RAG system?

ANSWER:
 A PDF-based RAG system extracts text from PDF documents, divides the text into smaller chunks, converts the chunks into numerical embeddings, and stores them in a vector database. When a user asks a question, the system retrieves the most relevant chunks and provides them to the language model as context.

RETRIEVED EVIDENCE:
 [Page 1]
provides that information to a language model to generate an answer. PDF-Based RAG A PDF-based RAG system extracts text from PDF documents, divides the text into smaller chunks, converts the chunks into numerical embeddings, and stores them in a vector database. When a user asks a question, the system retrieves the most relevant chunks and provides them to the language model as context. Advantages of RAG RAG can reduce the need to retrain a language model when documents change. It can also provide answers grounded in a specific collection of documents and can return references to the original document p

In [13]:
import re
import json

def extract_json(text):
    text = re.sub(r"```(?:json)?", "", text).strip()
    text = re.sub(r"```", "", text).strip()

    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        raise ValueError("No JSON object found in evaluator response.")

    return json.loads(match.group(0))


def evaluate_rag(question, answer, context):

    if not context.strip():
        return {
            "claims": [],
            "total_claims": 0,
            "supported_count": 0,
            "unsupported_count": 0,
            "hallucination_percentage": None,
            "status": "No evidence available"
        }

    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict RAG evaluator. Break the answer into "
                "individual factual claims. Compare each claim ONLY with "
                "the supplied context. A claim is supported if the context "
                "directly supports or clearly entails it. Otherwise mark it "
                "unsupported. Do not use outside knowledge. Return ONLY "
                "valid JSON in this format: "
                '{"claims":[{"claim":"text","supported":true,'
                '"evidence":"short explanation"}]}. '
                "The supported value must be true or false, not a string. "
                "Include all factual claims and do not omit unsupported claims."
            )
        },
        {
            "role": "user",
            "content": (
                f"QUESTION:\n{question}\n\n"
                f"CONTEXT:\n{context}\n\n"
                f"ANSWER TO EVALUATE:\n{answer}"
            )
        }
    ]

    raw_response = local_generate(
        messages,
        max_new_tokens=400
    )

    try:
        parsed = extract_json(raw_response)
        raw_claims = parsed.get("claims", [])

        if not isinstance(raw_claims, list):
            raise ValueError("Claims must be a list.")

        claims = []

        for item in raw_claims:
            if not isinstance(item, dict):
                continue

            claim_text = str(item.get("claim", "")).strip()

            if not claim_text:
                continue

            supported_value = item.get("supported", False)

            if isinstance(supported_value, bool):
                supported = supported_value
            elif isinstance(supported_value, str):
                supported = supported_value.strip().lower() == "true"
            else:
                supported = False

            claims.append({
                "claim": claim_text,
                "supported": supported,
                "evidence": str(item.get("evidence", "")).strip()
            })

    except (ValueError, json.JSONDecodeError, AttributeError) as error:
        return {
            "claims": [],
            "total_claims": 0,
            "supported_count": 0,
            "unsupported_count": 0,
            "hallucination_percentage": None,
            "status": f"Evaluation failed: {error}"
        }

    total = len(claims)
    supported_count = sum(c["supported"] for c in claims)
    unsupported_count = total - supported_count

    percentage = (
        round(unsupported_count / total * 100, 2)
        if total else None
    )

    return {
        "claims": claims,
        "total_claims": total,
        "supported_count": supported_count,
        "unsupported_count": unsupported_count,
        "hallucination_percentage": percentage,
        "status": "Completed" if total else "No claims extracted"
    }


print("Hallucination evaluator is ready!")

Hallucination evaluator is ready!


In [14]:
evaluation = evaluate_rag(
    result["question"],
    result["answer"],
    result["context"]
)

print("Evaluation status:", evaluation["status"])
print("Total claims:", evaluation["total_claims"])
print("Supported claims:", evaluation["supported_count"])
print("Unsupported claims:", evaluation["unsupported_count"])

if evaluation["hallucination_percentage"] is not None:
    print(
        "Estimated hallucination percentage:",
        f"{evaluation['hallucination_percentage']}%"
    )
else:
    print("Hallucination percentage unavailable.")

print("\nCLAIM DETAILS")

for claim in evaluation["claims"]:
    status = "SUPPORTED" if claim["supported"] else "UNSUPPORTED"

    print(f"\n[{status}] {claim['claim']}")
    print("Evidence:", claim["evidence"])

Evaluation status: Completed
Total claims: 1
Supported claims: 1
Unsupported claims: 0
Estimated hallucination percentage: 0.0%

CLAIM DETAILS

[SUPPORTED] PDF-based RAG system extracts text from PDF documents, divides the text into smaller chunks, converts the chunks into numerical embeddings, and stores them in a vector database.
Evidence: The provided definition states that a PDF-based RAG system extracts text from PDF documents, divides the text into smaller chunks, converts the chunks into numerical embeddings, and stores them in a vector database.


In [ ]:

import gradio as gr
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML

def run_dashboard(question, top_k):
    if not question or not question.strip():
        return (
            "Enter a question first.", "", [],
            "<p>Please enter a question.</p>", None
        )

    try:
        result = rag_answer(question.strip(), int(top_k))
        evaluation = evaluate_rag(
            result["question"],
            result["answer"],
            result["context"]
        )

        rows = []
        for c in evaluation["claims"]:
            rows.append([
                c["claim"],
                "SUPPORTED" if c["supported"] else "UNSUPPORTED",
                c["evidence"]
            ])

        pct = evaluation["hallucination_percentage"]
        pct_text = f"{pct:.1f}%" if pct is not None else "N/A"

        summary = f"""
        <div style="display:flex;gap:12px;flex-wrap:wrap;">
          <div style="background:#e8f5e9;color:#166534;padding:18px;border-radius:12px;min-width:120px;">
            <div>SUPPORTED</div>
            <h2>{evaluation['supported_count']}</h2>
          </div>
          <div style="background:#fee2e2;color:#991b1b;padding:18px;border-radius:12px;min-width:120px;">
            <div>UNSUPPORTED</div>
            <h2>{evaluation['unsupported_count']}</h2>
          </div>
          <div style="background:#dbeafe;color:#1e40af;padding:18px;border-radius:12px;min-width:150px;">
            <div>UNSUPPORTED RATE</div>
            <h2>{pct_text}</h2>
          </div>
        </div>
        <p>Total claims evaluated: {evaluation['total_claims']}</p>
        <p><b>Note:</b> This is an estimated unsupported-claim rate,
        not a guaranteed measure of hallucination.</p>
        """

        fig, ax = plt.subplots(figsize=(5, 4))
        supported = evaluation["supported_count"]
        unsupported = evaluation["unsupported_count"]

        if supported + unsupported > 0:
            ax.pie(
                [supported, unsupported],
                labels=["Supported", "Unsupported"],
                colors=["#22c55e", "#ef4444"],
                autopct="%1.1f%%",
                startangle=90
            )
            ax.set_title("Claim Support Analysis")
        else:
            ax.text(0.5, 0.5, "No claims to display",
                    ha="center", va="center")
            ax.axis("off")

        plt.tight_layout()

        return (
            result["answer"],
            result["context"],
            rows,
            summary,
            fig
        )

    except Exception as e:
        return (
            f"Error: {e}", "", [],
            "<p>Evaluation failed. Check the error.</p>", None
        )


with gr.Blocks(
    theme=gr.themes.Soft(),
    title="RAG Evaluation Dashboard"
) as demo:
    gr.Markdown("""
    # 🔎 RAG Evaluation & Hallucination Analysis
    ### Ask questions about your PDF and inspect answer reliability.
    """)

    with gr.Row():
        question_input = gr.Textbox(
            label="Your question",
            placeholder="What is a PDF-based RAG system?",
            lines=2,
            scale=4
        )
        top_k_input = gr.Slider(
            1, min(10, index.ntotal),
            value=min(5, index.ntotal),
            step=1,
            label="Retrieved chunks",
            scale=1
        )

    run_button = gr.Button(
        "🚀 Generate Answer & Evaluate",
        variant="primary"
    )

    answer_output = gr.Textbox(
        label="📝 Generated Answer",
        lines=5
    )
    context_output = gr.Textbox(
        label="📚 Retrieved PDF Evidence",
        lines=7
    )

    gr.Markdown("## 📊 Evaluation Results")
    summary_output = gr.HTML()

    with gr.Row():
        chart_output = gr.Plot(label="Claim Support Chart")

    claims_output = gr.Dataframe(
        headers=["Claim", "Status", "Evidence"],
        label="Claim-by-Claim Analysis",
        interactive=False,
        wrap=True
    )

    run_button.click(
        fn=run_dashboard,
        inputs=[question_input, top_k_input],
        outputs=[
            answer_output,
            context_output,
            claims_output,
            summary_output,
            chart_output
        ]
    )

demo.launch(share=True, debug=True)


/tmp/ipykernel_481/2773641466.py:87: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0e26283e0ef68efb8b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
